<a href="https://colab.research.google.com/github/dennymarcels/AreTheseTwoPeopleRelated/blob/master/nmt/%5BNMT%5D_Alimentar_child_manualDictionary_%5BRealtime%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Sobre
Objetivo(s) desse Notebook:

Atualizar o [child `manualDictionary` do Realtime do projeto Machine Learning](https://console.firebase.google.com/u/0/project/machinelearningexps/database/machinelearningexps-cdbg/data/~2Fnmt~2FmanualDictionary?hl=pt-br). Este child contém o dicionário de traduções da NMT, que é utilizado em algumas situações específicas sem utilizar a rede neural. Ele vai ser construído sempre a partir do zero.

# ⚙️ Instalações e Importações

In [1]:
from google.colab import auth, files, drive

# Autenticação Google
auth.authenticate_user()

# Montagem Drive
drive.mount('/content/drive')

# Chave de serviço com acesso ao Realtime do projeto Machine Learning
print('Faça o upload da chave de serviço do Machine Learning.')
ml_key_name = list(files.upload())[0]

Mounted at /content/drive
Faça o upload da chave de serviço do Machine Learning.


Saving key_firebase_ml.json to key_firebase_ml.json


Download de projeto(s)

In [2]:
# Download Pytools
!gcloud source repos clone github_hand-talk_pytools --project='ht-community'
!mv github_hand-talk_pytools pytools

$ git clone https://github.com/Hand-Talk/pytools
Cloning into '/content/github_hand-talk_pytools'...
remote: Total 1373 (delta 756), reused 1373 (delta 756)
Receiving objects: 100% (1373/1373), 757.70 KiB | 3.43 MiB/s, done.
Resolving deltas: 100% (756/756), done.
Project [ht-community] repository [github_hand-talk_pytools] was cloned to [/content/github_hand-talk_pytools].


Instalações

In [3]:
# Instalando Pytools
!pip install /content/pytools

Processing ./pytools
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 30.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 57.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 79.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.3/98.3 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.8/341.8 kB 34.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  P

## Importações

In [4]:
import pandas as pd
import cereja as cj

from handtalk.community import Community
from handtalk.gcloud import Firebase

🍒 Using Cereja v.1.8.9


# 📥 Parâmetros

In [5]:
#@title { run: "auto" }

#@markdown Selecione o ambiente de trabalho
WORKSPACE = "HT-BZS" #@param ["HT-BZS", "HT-ASL", "HT-Annoy"] {allow-input: false}

print('WORKSPACE:', WORKSPACE)

WORKSPACE: HT-BZS


# 🎯 Execução


Carregar dicionário legado.

In [13]:
legacy = pd.read_csv('/content/drive/Shareddrives/IA/CIENCIA_DE_DADOS/PROCESSOS/NMT/manual_dictionary_pt_bzs.csv', sep='\t')
legacy

,client,source_sentence,md5_sentence,target_sentence
0,Sicredi,O Programa de Capitalização das Cooperativas d...,7d9b2a8e4683024cd0fbf2d56f34352a,¶yYOTF07MPmxA8goNqUqW ¶cD3Gp1XtKmlb0hjXJCCn ¶c...
1,Sicredi,Linha de crédito que apoia você a adquirir e c...,5f6fd361f52a668ef11bcc4cb13f74cf,¶ioOm1rGs8YEpGEVDzDVG ¶0q91nydKpq7e7HnAkx8T ¶z...
2,Sicredi,Com essa linha de crédito você atende às suas ...,f24664d36ace6b8563e2ad78d244f2c3,¶ioOm1rGs8YEpGEVDzDVG ¶0q91nydKpq7e7HnAkx8T ¶G...
3,Sicredi,Facilitar a vida de quem coopera para o cresci...,49a4e0ecb94c2f4c82f505481df4e7e9,¶FGIiWMC2QnZpjCZ5jaTJ ¶3w6qpbImmBUKa8EYHJ0q ¶u...
4,Sicredi,"Com a linha de crédito Turismo, você realiza a...",3df58f7a2063ac64e0b435993f6583dd,¶ioOm1rGs8YEpGEVDzDVG ¶0q91nydKpq7e7HnAkx8T ¶x...
5,Sicredi,"Com o objetivo de cooperar e, juntos, investir...",e0af818a389bc50235541b7aeda2b627,¶0dFN0mvADWDXVSudzOVV ¶uM7FsH3yaRxbGFEUdC1k ¶R...
6,Claro,Cartão pré-pago recarregável,899a80c0cc5d41fc9c50444cc9777d94,¶ngeo7e7YyZjAqK4JljR5 ¶LutOYKAWJJJUCOuyhZzC ¶9...
7,Claro,Atendimento CIC em Libras,079c34dc07109cbb733bcd210605c6d4,cic ¶YGOfJK5TywJccj8lpZtA ¶uOSvoYcVeLehVUmUY9RZ
8,Claro,"Para atender com qualidade e eficiência, a cla...",69525e14253b9c8aecea815d1fb24557,¶cXBBoRD18vqs7AYJeAyd ¶YGOfJK5TywJccj8lpZtA ¶c...
9,Claro,"Paraatender com qualidade e eficiência, a clar...",1caa371b4f055997a23d28f834f5b90c,¶cXBBoRD18vqs7AYJeAyd ¶YGOfJK5TywJccj8lpZtA ¶c...


In [14]:
source = legacy['source_sentence'].tolist()
target = legacy['target_sentence'].tolist()
client = legacy['client'].tolist()
video = [None] * len(legacy)

Instanciar workspace.

In [ ]:
cmt = Community(WORKSPACE)

Os dados para o dicionário de traduções estarão disponíveis na coleção de `videos`, com origem `DICIONARIO_CLIENTES`.

In [ ]:
videos_col = cmt.list_longer_collection('videos', where_config=[('sentenceOrigin', '==', 'DICIONARIO_CLIENTES')])
len(videos_col)

🍒 Sys[out] » Total videos 13 query time 0.56s 
🍒 Progress » 000/100 - [▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱▱] - 0.00% - 🕜 00:00:01/nan estimated 

13

Coletando os dados para alimentar o dicionário

In [ ]:
for video_id, video_doc in videos_col.items():
    if video_doc.sentence.replace('\n', '') not in source:
        source.append(video_doc.sentence.replace('\n', ''))
        target.append(video_doc.translation.replace('\n', ''))
        client.append('Continental') #### <- a decidir
        video.append(video_id)

Visualizando dados.

In [ ]:
cj.visualize_sample(source, k=0, tr_k_iter=4, tr_k_str=0)

['O Programa de Capitalização das Cooperativas de Crédito (Procapsicredi) é '
 'uma linha de crédito que permite ao associado adquirir cotas-partes das '
 'Cooperativas do Sicredi.',
 'Linha de crédito que apoia você a adquirir e construir seu imóvel '
 'residencial novo ou usado.',
 '<…>',
 'Coifas e Depuradores',
 'Lançamentos Continental']


In [ ]:
cj.visualize_sample(target, k=0, tr_k_iter=4, tr_k_str=0)

['¶yYOTF07MPmxA8goNqUqW ¶cD3Gp1XtKmlb0hjXJCCn ¶cXBBoRD18vqs7AYJeAyd '
 '¶2RkFg7JdpHZ3dQjynb64 ¶0q91nydKpq7e7HnAkx8T procapsicredi '
 '¶ewOw724oLZGFaXft2R2z ¶ioOm1rGs8YEpGEVDzDVG ¶0q91nydKpq7e7HnAkx8T '
 '¶YdeNgFwXs7Nv4mGvHG7M ¶R4VwTpIXPQgBhEQDZH5t ¶1KIFXLaLDhagqISWbPRF cotas '
 '¶GoW8OuXTFFqHDa1aJLos ¶cXBBoRD18vqs7AYJeAyd ¶2RkFg7JdpHZ3dQjynb64 '
 '¶MojHSTThudafsxblqUDt',
 '¶ioOm1rGs8YEpGEVDzDVG ¶0q91nydKpq7e7HnAkx8T ¶zmCpTSq64NjT4DzJDrTS '
 '¶G8SC11YlrbGbxzzRJlBK ¶1KIFXLaLDhagqISWbPRF ¶RhUmg1264QsIqXGRXgij '
 '¶cKRR1C3spI0z1Yl23joO ¶rqXs9lr3E4s9awmxdhSS ¶xNVWidyEKSiXIN4CGfrA '
 '¶D8K9kkLRBJI0PkVezEYL ¶VvjPbF6M7fc9PsH2DsPW ¶JtBaDeeozLakXpg1cs7Z '
 '¶ajH8VB2qa1BwbUNYESxr',
 '<…>',
 '¶CTgi4oO51YdJl9d4kULJ ¶YgRucgnLKxucJDpxq0AE ¶RhUmg1264QsIqXGRXgij '
 '¶CTgi4oO51YdJl9d4kULJ ¶OmUvF2Njjlx3aNbflaVt ¶D8K9kkLRBJI0PkVezEYL',
 '¶l6cKc4jpHLDJCeESM417 ¶cXBBoRD18vqs7AYJeAyd Continental']


In [ ]:
cj.visualize_sample(client, k=0, tr_k_iter=4, tr_k_str=0)

['Sicredi', 'Sicredi', '<…>', 'Continental', 'Continental']


In [ ]:
cj.visualize_sample(video, k=0, tr_k_iter=4, tr_k_str=0)

[None, None, '<…>', 'NCiPBfAeDtgWuQhb9JAW', 'uhKIA8PoZrNom6T3bJz0']


Montando update

In [ ]:
update = [{'source': s, 'target': t, 'client': c, 'video': v} for s, t, c, v in zip(source, target, client, video)]
cj.visualize_sample(update, k=0, tr_k_iter=4, tr_k_str=30)

[{'client': 'Sicredi',
  'source': 'O Programa de C…vas do Sicredi.',
  'target': '¶yYOTF07MPmxA8g…TThudafsxblqUDt',
  'video': None},
 {'client': 'Sicredi',
  'source': 'Linha de crédit… novo ou usado.',
  'target': '¶ioOm1rGs8YEpGE…B2qa1BwbUNYESxr',
  'video': None},
 '<…>',
 {'client': 'Continental',
  'source': 'Coifas e Depuradores',
  'target': '¶CTgi4oO51YdJl9…kLRBJI0PkVezEYL',
  'video': 'NCiPBfAeDtgWuQhb9JAW'},
 {'client': 'Continental',
  'source': 'Lançamentos Continental',
  'target': '¶l6cKc4jpHLDJCe…Ayd Continental',
  'video': 'uhKIA8PoZrNom6T3bJz0'}]


Instanciando Realtime

In [ ]:
realtime = Firebase(service_key=ml_key_name,
                    project_id='machinelearningexps-cdbg',
                    bucket_name='ht-nmt',
                    app_name='ml').realtime

Carregar dados no Realtime.

In [ ]:
workspace = WORKSPACE.split('-')[1].lower()

if input('Deseja atualizar o Realtime? Ss/[Nn]').lower() == 's':
    realtime.db.child(f'nmt/manualDictionary/{workspace}').set(update)

Deseja atualizar o Realtime? Ss/[Nn]s


Conferir última entrada do dicionário.

In [ ]:
realtime.db.child(f'nmt/manualDictionary/{workspace}').get()[-1]

{'client': 'Continental',
 'source': 'Lançamentos Continental',
 'target': '¶l6cKc4jpHLDJCeESM417 ¶cXBBoRD18vqs7AYJeAyd Continental',
 'video': 'uhKIA8PoZrNom6T3bJz0'}

⭑･ﾟﾟ･*:༅｡.｡༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･･*:༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･*:༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･⭑ [̲̅F̲̅I̲̅м̲̅] ⭑･ﾟﾟ･*:༅｡.｡༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･･*:༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･*:༅:*ﾟ:*:✼✿✼:*ﾟ:༅:*･ﾟﾟ･⭑